# **Cross Validation**

## 1 Objetivo

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.


**Descrição da Base de Dados**
A base de dados contém as seguintes variáveis:

- `Unnamed:0`: Índice (não é uma variável útil para o modelo)

- `UTC`: Tempo em Segundos UTC

- `Temperature[C]`: Temperatura do Ar (em graus Celsius)

- `Humidity[%]`: Umidade do Ar (em porcentagem)

- `TVOC[ppb]`: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

- `eCO2[ppm]`: Concentração equivalente de CO2 (medido em partes por milhão)

- `Raw H2`: Hidrogênio molecular bruto, não compensado

- `Raw Ethanol`: Etanol gasoso bruto

- `Pressure[hPA]`: Pressão do Ar (em hectopascais)

- `PM1.0`: Material particulado de tamanho < 1,0 µm

- `PM2.5`: Material particulado de tamanho >1,0 µm e < 2,5 µm

- `NC0.5`: Concentração numérica de material particulado de tamanho < 0,5 µm

- `NC1.0`: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

- `NC2.5`: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

- `CNT`: Contador de amostras


E a variável alvo:

- `Fire Alarm`: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import StratifiedKFold, cross_val_score

## 2 Carregamento da base de dados

In [2]:
df = pd.read_csv('Cientista de dados M35 - smoke_detection_iot.csv')
df.head()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


## 3 Verificação inicial dos dados

In [3]:
# Verificando a quantidade de linhas e colunas
df.shape

(62630, 16)

In [4]:
# Verificando os tipos de dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      62630 non-null  int64  
 1   UTC             62630 non-null  int64  
 2   Temperature[C]  62630 non-null  float64
 3   Humidity[%]     62630 non-null  float64
 4   TVOC[ppb]       62630 non-null  int64  
 5   eCO2[ppm]       62630 non-null  int64  
 6   Raw H2          62630 non-null  int64  
 7   Raw Ethanol     62630 non-null  int64  
 8   Pressure[hPa]   62630 non-null  float64
 9   PM1.0           62630 non-null  float64
 10  PM2.5           62630 non-null  float64
 11  NC0.5           62630 non-null  float64
 12  NC1.0           62630 non-null  float64
 13  NC2.5           62630 non-null  float64
 14  CNT             62630 non-null  int64  
 15  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(8)
memory usage: 7.6 MB


In [5]:
# Verificando se existem dados nulos
df.isnull().sum()

Unnamed: 0        0
UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire Alarm        0
dtype: int64

In [6]:
# Dados estatístico do conjunto de dados
df.describe()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
count,62630.000000,6.263000e+04,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000
mean,31314.500000,1.654792e+09,15.970424,48.539499,1942.057528,670.021044,12942.453936,19754.257912,938.627649,100.594309,184.467770,491.463608,203.586487,80.049042,10511.386157,0.714626
std,18079.868017,1.100025e+05,14.359576,8.865367,7811.589055,1905.885439,272.464305,609.513156,1.331344,922.524245,1976.305615,4265.661251,2214.738556,1083.383189,7597.870997,0.451596
min,0.000000,1.654712e+09,-22.010000,10.740000,0.000000,400.000000,10668.000000,15317.000000,930.852000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15657.250000,1.654743e+09,10.994250,47.530000,130.000000,400.000000,12830.000000,19435.000000,938.700000,1.280000,1.340000,8.820000,1.384000,0.033000,3625.250000,0.000000
50%,31314.500000,1.654762e+09,20.130000,50.150000,981.000000,400.000000,12924.000000,19501.000000,938.816000,1.810000,1.880000,12.450000,1.943000,0.044000,9336.000000,1.000000
75%,46971.750000,1.654778e+09,25.409500,53.240000,1189.000000,438.000000,13109.000000,20078.000000,939.418000,2.090000,2.180000,14.420000,2.249000,0.051000,17164.750000,1.000000
max,62629.000000,1.655130e+09,59.930000,75.200000,60000.000000,60000.000000,13803.000000,21410.000000,939.861000,14333.690000,45432.260000,61482.030000,51914.680000,30026.438000,24993.000000,1.000000


In [7]:
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)
df.head()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire_Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [8]:
df["Fire_Alarm"].unique()

array([0, 1])

Na verificação inicial, observa-se que a base possui 62.630 registros e 16 colunas. Todas as colunas apresentam dados completos, sem valores nulos ou ausentes.

Os tipos de dados também estão adequados para a modelagem, pois as variáveis são numéricas, compostas por valores inteiros e decimais. A variável alvo Fire Alarm possui apenas os valores 0 e 1, indicando que se trata de um problema de classificação binária.

A coluna Unnamed: 0 representa apenas um índice da base e, por isso, será removida antes da etapa de modelagem, pois não contribui para a previsão da ocorrência de incêndio.

## 4 Escolha do modelo



Para esta atividade, será utilizado o modelo de Regressão Logística, pois o problema consiste em uma tarefa de classificação binária, em que a variável alvo Fire_Alarm indica se houve ou não ocorrência de incêndio.

A Regressão Logística é um modelo adequado para esse tipo de problema, pois estima a probabilidade de uma observação pertencer a uma determinada classe. Além disso, é um modelo simples, interpretável e bastante utilizado como baseline em problemas de classificação.

Como a base possui variáveis numéricas em diferentes escalas, será utilizado o StandardScaler dentro de um Pipeline para padronizar os dados antes do treinamento do modelo.

## 5 Separação da base em Y e X 

In [9]:
df.drop('Unnamed: 0', axis=1, inplace=True, errors='ignore')

In [10]:
X = df.drop(columns=['Fire_Alarm'])
y = df['Fire_Alarm']

modelo = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42))
])

In [11]:
X.head()

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT
0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0
1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1
2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2
3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3
4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4


In [12]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Fire_Alarm, dtype: int64

##  6 Validação cruzada

In [13]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    modelo,
    X,
    y,
    cv=kfold,
    scoring='accuracy'
)

scores

array([0.98946192, 0.98874341, 0.99002076, 0.98954175, 0.99065943])

In [14]:
print(f'Acurácia média: {scores.mean() * 100:.2f}%')
print(f'Desvio padrão: {scores.std() * 100:.2f}%')

Acurácia média: 98.97%
Desvio padrão: 0.06%


A validação cruzada foi aplicada com 5 folds, permitindo avaliar o desempenho do modelo em diferentes divisões da base de dados. A pontuação obtida em cada fold demonstra como o modelo se comporta em diferentes subconjuntos dos dados.

A acurácia média representa o desempenho geral do modelo após a validação cruzada. Como a média das pontuações foi calculada a partir dos diferentes folds, essa avaliação é mais robusta do que utilizar apenas uma única separação entre treino e teste.

# 7 Avaliação da pontuação de cada fold e validação final da média

In [15]:
for i, score in enumerate(scores, start=1):
    print(f'Fold {i}: {score * 100:.2f}%')

print(f'\nAcurácia média: {scores.mean() * 100:.2f}%')
print(f'Desvio padrão: {scores.std() * 100:.2f}%')

Fold 1: 98.95%
Fold 2: 98.87%
Fold 3: 99.00%
Fold 4: 98.95%
Fold 5: 99.07%

Acurácia média: 98.97%
Desvio padrão: 0.06%


A validação cruzada foi aplicada utilizando 5 folds. Em cada divisão, o modelo foi treinado em uma parte dos dados e avaliado em outra, permitindo verificar seu desempenho em diferentes subconjuntos da base.

As pontuações obtidas em cada fold ficaram muito próximas, indicando que o modelo apresentou desempenho estável. A acurácia média foi de aproximadamente 98,97%, com desvio padrão de 0,06%, o que demonstra uma boa capacidade de generalização do modelo para os dados avaliados.

Dessa forma, a Regressão Logística, combinada com a padronização das variáveis por meio do StandardScaler, apresentou um desempenho satisfatório para prever a variável Fire_Alarm.